# 06 · Controle: sentimento sintético

A etapa 05 mostrou que o sentimento real não melhora as redes. Este controle verifica se as redes **conseguiriam** usar um sentimento que antecipasse o mercado.

Cada versão soma ao OHLCV uma coluna `Sinal = tanh(ρ·z + √(1−ρ²)·ruído)`:
- **amanha_ρ:** z é o retorno padronizado de **amanhã**. ρ = 1 é um sinal perfeito; ρ menor mistura mais ruído.
- **hoje:** z é o retorno de **hoje**, como o sentimento real faz. É o controle negativo.

Os hiperparâmetros são os das etapas 02 e 03, com 5 sementes. A versão financeira é a da etapa 05.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from tcc import config, controles, dados, experimento

In [ ]:
ALVO   = 'retorno'
INICIO = '2022-01-01'
FIM    = '2023-12-31'
MOEDAS = config.MOEDAS
VARIANTES = {'hoje': ('hoje', 1.0), 'amanha_1.0': ('amanha', 1.0), 'amanha_0.5': ('amanha', 0.5),
             'amanha_0.25': ('amanha', 0.25), 'amanha_0.1': ('amanha', 0.1)}

pasta = lambda etapa: os.path.join(RAIZ, 'results', etapa)
MODELOS = {'LSTM': json.load(open(os.path.join(pasta('02_lstm'), 'hiperparametros.json'))),
           'Transformer': json.load(open(os.path.join(pasta('03_transformer'), 'hiperparametros.json')))}
ETAPA_05 = pd.read_csv(os.path.join(pasta('05_com_sentimento'), 'previsoes.csv'), parse_dates=['data'])

SAIDA = config.PASTA_RESULTADOS / '06_controle'
SAIDA.mkdir(parents=True, exist_ok=True)

## O sinal sintético

Spearman de cada sinal com o retorno de hoje e de amanhã, nos dias de teste.

In [ ]:
sinais, linhas = {}, []
for m in MOEDAS:
    _, r = dados.entradas(dados.precos(m, INICIO, FIM), ALVO)
    teste = r.index[experimento.cortes(len(r))[1]:-1]
    for v, (quando, rho) in VARIANTES.items():
        sinais[m, v] = controles.sinal(m, INICIO, FIM, rho, quando)
        s = sinais[m, v]['Sinal']
        linhas.append({'moeda': m, 'variante': v,
                       'rho_hoje': spearmanr(s[teste], r[teste])[0],
                       'rho_amanha': spearmanr(s[teste], r.shift(-1)[teste])[0]})
correlacao = pd.DataFrame(linhas)
correlacao.round(4).to_csv(SAIDA / 'correlacao_sinal.csv', index=False)
correlacao.groupby('variante', sort=False)[['rho_hoje', 'rho_amanha']].mean().round(3)

## Treino e teste

In [ ]:
partes = [ETAPA_05[(ETAPA_05.variante == 'financeiro') & ETAPA_05.moeda.isin(MOEDAS)]]
for m in MOEDAS:
    for v in VARIANTES:
        for nome, hp in MODELOS.items():
            partes.append(experimento.rodar(m, ALVO, INICIO, FIM, hp=hp, extras=sinais[m, v]).assign(modelo=nome, variante=v))
        print(m, v, 'ok')
prev = pd.concat(partes, ignore_index=True)
prev.to_csv(SAIDA / 'previsoes.csv', index=False)

## MAPE (%)

In [ ]:
ordem = ['financeiro', *VARIANTES]
resumo = experimento.resumir(prev, chave=['modelo', 'variante'])
resumo.round(4).to_csv(SAIDA / 'resumo.csv', index=False)
tabela = resumo.pivot_table(index='moeda', columns=['modelo', 'variante'], values='MAPE_%').loc[MOEDAS]
tabela = tabela[[(mo, v) for mo in MODELOS for v in ordem]]
tabela.insert(0, ('naive', ''), resumo.groupby('moeda')['MAPE_naive_%'].first())
tabela.loc['média'] = tabela.mean()
tabela.round(3).to_csv(SAIDA / 'mape.csv')
tabela.round(3)

## Diebold-Mariano: cada sinal x financeiro

DM < 0: a versão com o sinal erra menos. p < 0,05: diferença significativa.

In [ ]:
dm = pd.concat([experimento.comparar(prev[prev.modelo == nome], 'financeiro', 'variante', list(VARIANTES)).assign(modelo=nome)
                for nome in MODELOS], ignore_index=True)
dm.round(4).to_csv(SAIDA / 'diebold_mariano.csv', index=False)
dm.pivot_table(index='moeda', columns=['modelo', 'variante'], values='p').loc[MOEDAS].round(3)

## Acerto de direção (%)

In [ ]:
resumo.pivot_table(index='moeda', columns=['modelo', 'variante'], values='acerto_direcao_%').loc[MOEDAS].round(1)

In [ ]:
media = resumo.groupby(['modelo', 'variante'])[['MAPE_%', 'MAPE_naive_%']].mean()
rhos = [VARIANTES[v][1] for v in VARIANTES if v.startswith('amanha')]
fig, ax = plt.subplots(figsize=(7, 4))
for nome in MODELOS:
    ax.plot(rhos, [media.loc[(nome, v), 'MAPE_%'] for v in VARIANTES if v.startswith('amanha')], 'o-', label=f'{nome} com sinal de amanhã')
    ax.axhline(media.loc[(nome, 'financeiro'), 'MAPE_%'], ls=':', color=ax.lines[-1].get_color(), label=f'{nome} financeiro')
ax.axhline(media['MAPE_naive_%'].iloc[0], color='r', ls='--', label='naive')
ax.set_xscale('log'); ax.set_xticks(rhos); ax.set_xticklabels(rhos)
ax.set_xlabel('ρ do sinal com o retorno de amanhã'); ax.set_ylabel('MAPE médio das 5 moedas (%)'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(SAIDA / 'mape_por_rho.png', dpi=130); plt.show()

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'alvo': ALVO, 'inicio': INICIO, 'fim': FIM, 'moedas': MOEDAS,
    'sementes': config.SEMENTES, 'variantes': VARIANTES, 'semente_ruido': 42, 'hp': MODELOS}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))